# Generador de Pokémon con una GAN

Entrenamiento de una DCGAN sobre sprites de Pokémon, celda a celda.

La documentación del proyecto (cómo funciona, qué se rompió por el camino, los
resultados y las referencias) está en el **README** del repositorio. Aquí solo
queda lo que se ejecuta.

## 1. Configuración

Rutas, tamaño de lote, dimensión del espacio latente y número de épocas.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import os
import matplotlib.pyplot as plt

# --- 1. CONFIGURACIÓN ---
OUTPUT_DIR = "entrenamiento_pokemon_10000"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

DATA_PATH = "pokemon/pokemon_jpg/pokemon_jpg" 
BATCH_SIZE = 64 
noise_dim = 100
EPOCHS = 10000

## 2. Dataset

Unas 800 imágenes, redimensionadas a 64x64 y normalizadas al rango [-1, 1],
que es el que devuelve la `tanh` del generador. `prefetch` solapa la carga con
el cómputo.

In [ ]:
# Limpieza inicial
tf.keras.backend.clear_session()

# --- 2. DATASET ---
def preprocess(img):
    return (tf.cast(img, tf.float32) - 127.5) / 127.5

print("Cargando dataset...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_PATH, label_mode=None, image_size=(64, 64), batch_size=BATCH_SIZE)
train_dataset = train_dataset.map(preprocess).shuffle(1000).prefetch(tf.data.AUTOTUNE)

## 3. Los modelos

El generador va de un vector de 100 dimensiones a una imagen de 64x64x3 con
tres `Conv2DTranspose`. El discriminador es un clasificador binario con dos
`Conv2D`, `LeakyReLU` y `Dropout(0.3)`.

Los dos optimizadores usan ritmos distintos a propósito (1e-4 el generador,
2e-5 el discriminador), y la pérdida lleva label smoothing de 0.9. El porqué de
esas tres decisiones está explicado en el README.

In [ ]:
# --- 3. MODELOS ---
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(noise_dim,)),
        layers.Dense(8*8*256, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((8, 8, 256)),
        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2DTranspose(3, 5, strides=2, padding="same", use_bias=False, activation="tanh")
    ])
    return model

def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(64, 64, 3)),
        layers.Conv2D(64, 5, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Conv2D(128, 5, strides=2, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [ ]:
generator = build_generator()
discriminator = build_discriminator()

# --- 4. OPTIMIZADORES Y PÉRDIDA ---
generator_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-5, beta_1=0.5)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

seed = tf.random.normal([16, noise_dim])

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([images.shape[0], noise_dim])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = cross_entropy(tf.ones_like(fake_output), fake_output)
        # Label smoothing para evitar que el discriminador sea demasiado agresivo
        real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)
        fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
        disc_loss = real_loss + fake_loss

    generator_optimizer.apply_gradients(zip(gen_tape.gradient(gen_loss, generator.trainable_variables), generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(disc_tape.gradient(disc_loss, discriminator.trainable_variables), discriminator.trainable_variables))
    

## 4. Entrenamiento

Se guarda una rejilla de muestras cada 20 épocas durante las primeras 100 y
cada 100 a partir de ahí, más un checkpoint del generador cada 500.

In [ ]:
def save_and_plot(epoch):
    imgs = generator(seed, training=False)
    fig = plt.figure(figsize=(6,6))
    for i in range(16):
        plt.subplot(4, 4, i+1)
        plt.imshow((imgs[i] + 1) / 2)
        plt.axis('off')
    plt.savefig(f"{OUTPUT_DIR}/epoch_{epoch+1:05d}.png")
    plt.close()

In [ ]:
# --- 5. LOOP DE ENTRENAMIENTO ---
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("modelo", exist_ok=True)

print(f"Iniciando entrenamiento de {EPOCHS} épocas...")

for epoch in range(EPOCHS):
    for image_batch in train_dataset:
        train_step(image_batch)
    
    current_epoch = epoch + 1
    
    # Lógica de visualización
    # 1. Los primeros 100 cada 20
    if current_epoch <= 100:
        if current_epoch % 20 == 0:
            print(f"Fase inicial: Época {current_epoch} guardada.")
            save_and_plot(epoch)
    
    # 2. A partir de 100 cada 100
    else:
        if current_epoch % 100 == 0:
            print(f"Fase larga: Época {current_epoch} guardada.")
            save_and_plot(epoch)
            
    # Guardar el modelo cada 500 épocas (backup de seguridad)
    if current_epoch % 500 == 0:
        generator.save(f"checkpoints/gen_checkpoint_{current_epoch}.h5")

# GUARDAR FINAL
generator.save("modelo/generador_pokemon_10000_final.h5")
print("Entrenamiento finalizado y modelo guardado")